# **VLMs Evaluation**



In order to make this comparison we actually had to convert the ground-truth data, which was a CSV (since that's the output that comes from Label Studio) to `json` formats. We used the `df_structure_to_json.py` script for this task

In [ ]:
resume_images_directory = "data/resumes_images"

In [ ]:
import pandas as pd
from utils.helpers import FormateLSOutputReadingOrder, OrganizeLSOutput

ground_truth_data = pd.read_csv("/data/ground_truth_LS_112.csv")

ground_truth_data = FormateLSOutputReadingOrder(ground_truth_data).process()

ground_truth_data = OrganizeLSOutput(
    ground_truth_data,
    transcription_column="ro_transcription",
    rename={"words": "words_ro"},
    images_directory=resume_images_directory
).process_all()

In [ ]:
from utils import df_to_structure_json

df_to_structure_json(ground_truth_data, output_dir="structured_jsons_GT_2")

In [ ]:
import unicodedata, re, json, jiwer
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
SECTION_MAP = {
    "PERSONAL":  "personal_info",
    "SKILLS":    "skills",
    "EDUCATION": "education",
    "EMP-INFO":  "work_experience_info",
    "EMP-DESC":  "work_experience_description",
}

EMPTY_PLACEHOLDERS = {"n/a", "na", "none", "null", "[]", "{}", '""', "''", "-", "--"}

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
# ------------ Text utilities ----------- 

def normalize_text(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def flatten(value) -> str:
    if isinstance(value, str):   return value
    if isinstance(value, list):  return " ".join(flatten(v) for v in value)
    if isinstance(value, dict):  return " ".join(flatten(v) for v in value.values())
    return str(value) if value is not None else ""

def is_empty(text) -> bool:
    return not text or str(text).strip().lower() in EMPTY_PLACEHOLDERS

In [ ]:
# ------------ Extraction -----------

def extract_gt_text(gt_json: dict) -> dict:
    return {
        section: normalize_text(" ".join(
            item["transcription"] for item in items if "transcription" in item
        ))
        for section, items in gt_json.get("section", {}).items()
    }

def extract_gt_text_multi(gt_json_list: list) -> dict:
    aggregated = defaultdict(str)
    for gt_json in gt_json_list:
        for section, text in extract_gt_text(gt_json).items():
            aggregated[section] = (aggregated[section] + " " + text).strip()
    return {s: normalize_text(t) for s, t in aggregated.items()}

def extract_pred_text(pred_json: dict) -> dict:
    extracted = {}
    for key in ("personal_info", "skills", "education"):
        if key in pred_json:
            extracted[key] = normalize_text(flatten(pred_json[key]))

    if "work_experience" in pred_json:
        work_exp = pred_json["work_experience"]
        exps = [work_exp] if isinstance(work_exp, dict) else (work_exp if isinstance(work_exp, list) else [])
        info_parts, desc_parts = [], []
        for exp in exps:
            if not isinstance(exp, dict):
                continue
            desc_parts.append(flatten(exp.get("employment_description", "")))
            info_parts.extend(flatten(v) for k, v in exp.items() if k != "employment_description")
        extracted["work_experience_info"]        = normalize_text(" ".join(info_parts))
        extracted["work_experience_description"] = normalize_text(" ".join(desc_parts))

    return extracted

In [ ]:
# ------------ Metrics -----------

def compute_wer(gt: str, pred: str) -> float:
    if not gt: return 0.0 if not pred else 1.0
    return jiwer.wer(gt, pred)

def semantic_similarity(a: str, b: str) -> float:
    if not a.strip() or not b.strip(): return 0.0
    emb = model.encode([a, b], normalize_embeddings=True)
    return float(cosine_similarity(emb[0:1], emb[1:2])[0][0])

In [ ]:
# ------------ Accumulator helpers -----------

def _empty_acc():
    return {"wer_sum": 0.0, "sem_sim_sum": 0.0, "wer_values": [],
            "sem_sim_values": [], "count": 0, "gt_tokens": 0, "pred_tokens": 0}

def _accumulate(acc: dict, wer: float, sem_sim: float, gt: str, pred: str):
    acc["wer_sum"]       += wer
    acc["sem_sim_sum"]   += sem_sim
    acc["wer_values"].append(round(wer, 4))
    acc["sem_sim_values"].append(round(sem_sim, 4))
    acc["count"]         += 1
    acc["gt_tokens"]     += len(str(gt).split())
    acc["pred_tokens"]   += len(str(pred).split())

def _finalize(acc: dict) -> dict:
    n = acc["count"]
    return {
        "avg_semantic_similarity":   round(acc["sem_sim_sum"] / n, 4),
        "avg_wer":                   round(acc["wer_sum"] / n, 4),
        "semantic_similarity_values": acc["sem_sim_values"],
        "wer_values":                acc["wer_values"],
        "gt_tokens":                 acc["gt_tokens"],
        "pred_tokens":               acc["pred_tokens"],
        "brevity_penalty":           round(min(1.0, acc["pred_tokens"] / max(1, acc["gt_tokens"])), 4),
    }

In [ ]:
def compare_json_files(gt_paths, pred_path, debug=False, empty_gt_policy="skip"):
    """Accepts a single gt_path or a list of gt_paths (multi-page)."""
    gt_paths  = [gt_paths] if not isinstance(gt_paths, list) else gt_paths
    gt_jsons  = [load_json(p) for p in gt_paths]
    pred_json = load_json(pred_path)

    gt_sections   = extract_gt_text_multi(gt_jsons) if len(gt_jsons) > 1 else extract_gt_text(gt_jsons[0])
    pred_sections = extract_pred_text(pred_json)

    results = {}
    for gt_section, pred_section in SECTION_MAP.items():
        gt_text, pred_text = gt_sections.get(gt_section, ""), pred_sections.get(pred_section, "")
        gt_empty, pred_empty = is_empty(gt_text), is_empty(pred_text)

        if gt_empty and pred_empty:
            if debug: print(f"[SKIP] Both empty for '{gt_section}'")
            continue

        if gt_empty:
            if   empty_gt_policy == "skip":     continue
            elif empty_gt_policy == "soft":     sem_sim, wer = 1.0, 0.0
            elif empty_gt_policy == "penalize": sem_sim, wer = 0.0, 1.0
            else: raise ValueError("empty_gt_policy must be 'skip', 'soft', or 'penalize'")
            if debug: print(f"[{empty_gt_policy.upper()}] GT empty for '{gt_section}'")
        else:
            sem_sim = semantic_similarity(gt_text, pred_text)
            wer     = compute_wer(gt_text, pred_text)

        acc = results.setdefault(pred_section, _empty_acc())
        _accumulate(acc, wer, sem_sim, gt_text, pred_text)

    return {s: _finalize(v) for s, v in results.items()}

In [ ]:
def load_json(path) -> dict:
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(f"File not found: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def group_gt_files_by_cv(gt_files) -> dict:
    groups = defaultdict(list)
    for f in gt_files:
        groups[re.sub(r"_page_\d+$", "", f.stem)].append(f)
    return {k: sorted(v) for k, v in groups.items()}

def compare_json_directories(gt_dir, pred_dir, multi_gt=False, debug=False, empty_gt_policy="skip"):
    gt_dir, pred_dir = Path(gt_dir), Path(pred_dir)
    gt_files = sorted(gt_dir.glob("*.json"))

    pairs = (
        {cv: (pages, pred_dir / f"{cv}.json") for cv, pages in group_gt_files_by_cv(gt_files).items()}
        if multi_gt else
        {f.stem: ([f], pred_dir / f.name) for f in gt_files}
    )

    all_results, global_acc, skipped = {}, defaultdict(lambda: {
        "sem_sim_sum": 0.0, "wer_sum": 0.0, "count": 0, "gt_tokens": 0, "pred_tokens": 0
    }), []

    for name, (gt_paths, pred_file) in pairs.items():
        if not pred_file.exists():
            skipped.append(name); continue

        file_result = compare_json_files(gt_paths, pred_file, debug=debug, empty_gt_policy=empty_gt_policy)
        all_results[name] = file_result

        for section, metrics in file_result.items():
            g = global_acc[section]
            g["sem_sim_sum"] += metrics["avg_semantic_similarity"]
            g["wer_sum"]     += metrics["avg_wer"]
            g["count"]       += 1
            g["gt_tokens"]   += metrics["gt_tokens"]
            g["pred_tokens"] += metrics["pred_tokens"]

    if debug:
        print(f"\nMatched: {len(all_results)}  Skipped: {len(skipped)}")
        if skipped: print(f"Skipped: {skipped}")

    global_results = {
        s: {
            "avg_semantic_similarity": round(v["sem_sim_sum"] / max(1, v["count"]), 4),
            "avg_wer":                 round(v["wer_sum"]     / max(1, v["count"]), 4),
            "gt_tokens":               v["gt_tokens"],
            "pred_tokens":             v["pred_tokens"],
            "brevity_penalty":         round(min(1.0, v["pred_tokens"] / max(1, v["gt_tokens"])), 4),
            "contributing_cvs":        v["count"],
        }
        for s, v in global_acc.items()
    }

    key = "per_cv" if multi_gt else "per_file"
    return {key: all_results, "global": global_results}

In [ ]:
# Stage 1 — single GT per file
results = compare_json_directories(gt_dir="...", pred_dir="...")
print(json.dumps(results["global"], indent=2))

# Stage 2 — multi-page GT
results = compare_json_directories(gt_dir="...", pred_dir="...", multi_gt=True, debug=True)